In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
merged_df = pd.read_parquet("../Model_Data/merged_df_50Hz.parquet")

In [3]:
from itertools import combinations

TEST_FRACTION = 0.2

# Author pro ID
author_per_id = merged_df.groupby("ID")["Author"].first()
ids_per_class = merged_df.groupby("Tag")["ID"].unique().to_dict()

train_ids = []
test_ids  = []
split_log = []

for tag, ids in ids_per_class.items():
    ids     = np.array(ids)
    authors = author_per_id.loc[ids].values

    unique_authors = sorted(set(authors))
    author_ids     = {a: ids[authors == a] for a in unique_authors}
    counts         = {a: len(v) for a, v in author_ids.items()}
    total          = len(ids)
    target_test    = TEST_FRACTION * total

    # Alle 2^n Untermengen der Authors durchgehen, die mit Test-Anteil
    # am naehesten an 20% beste waehlen. Tiebreak: kleinere Untermenge.
    best_subset = ()
    best_key    = (float("inf"), float("inf"))   # (|diff|, |subset|)
    for r in range(len(unique_authors) + 1):
        for subset in combinations(unique_authors, r):
            sum_test = sum(counts[a] for a in subset)
            key = (abs(sum_test - target_test), len(subset))
            if key < best_key:
                best_key    = key
                best_subset = subset

    test_authors = set(best_subset)
    for a, id_array in author_ids.items():
        if a in test_authors:
            test_ids.extend(id_array.tolist())
        else:
            train_ids.extend(id_array.tolist())

    test_count = sum(counts[a] for a in test_authors)
    split_log.append({
        "Tag":           tag,
        "Total_IDs":     total,
        "Train_IDs":     total - test_count,
        "Test_IDs":      test_count,
        "Test_%":        round(test_count / total * 100, 1),
        "Test_Authors":  ", ".join(sorted(test_authors)) or "-",
        "Train_Authors": ", ".join(sorted(set(unique_authors) - test_authors)) or "-",
    })

# Train / Test DataFrames
df_train = merged_df[merged_df["ID"].isin(train_ids)].copy()
df_test  = merged_df[merged_df["ID"].isin(test_ids)].copy()

n_ids = merged_df["ID"].nunique()
print(f"Train shape: {df_train.shape}")
print(f"Test  shape: {df_test.shape}")
print(f"Test fraction (rows): {len(df_test) / len(merged_df):.1%}")
print(f"Test fraction (IDs):  {len(test_ids) / n_ids:.1%}")
print()
print("Author-disjunkter Split pro Tag:")
print(pd.DataFrame(split_log).set_index("Tag"))
print()
print("Test-IDs pro Tag x Author:")
print(df_test.groupby(["Tag", "Author"])["ID"].nunique().unstack(fill_value=0))
print()
print("Train-IDs pro Tag x Author:")
print(df_train.groupby(["Tag", "Author"])["ID"].nunique().unstack(fill_value=0))

Train shape: (1107963, 17)
Test  shape: (210811, 17)
Test fraction (rows): 16.0%
Test fraction (IDs):  19.7%

Author-disjunkter Split pro Tag:
           Total_IDs  Train_IDs  Test_IDs  Test_%                Test_Authors  \
Tag                                                                             
Auto             144        117        27    18.8                      Renate   
Laufen           136        105        31    22.8                      Renate   
Lift             118        100        18    15.3          Raphael_Schuepbach   
Roundkick         74         59        15    20.3              Renate, Tobias   
Treppe           107         75        32    29.9                      Renate   
Velo             119        105        14    11.8              Jessica_Schmid   
Zug              129        103        26    20.2  Raphael_Schuepbach, Renate   

                                               Train_Authors  
Tag                                                           
A

In [4]:
def split_distribution(df_train, df_test, level="ID"):
    """level='ID' -> Anzahl Mess-IDs, level='rows' -> Anzahl Zeilen."""
    if level == "ID":
        train = df_train.groupby("Tag")["ID"].nunique()
        test  = df_test.groupby("Tag")["ID"].nunique()
        col_train, col_test = "Train_IDs", "Test_IDs"
    else:
        train = df_train.groupby("Tag").size()
        test  = df_test.groupby("Tag").size()
        col_train, col_test = "Train_Rows", "Test_Rows"

    out = pd.DataFrame({col_train: train, col_test: test}).fillna(0).astype(int)
    out["Total"]   = out[col_train] + out[col_test]
    out["Train_%"] = (out[col_train] / out["Total"] * 100).round(1)
    out["Test_%"]  = (out[col_test]  / out["Total"] * 100).round(1)

    total_train = int(out[col_train].sum())
    total_test  = int(out[col_test].sum())
    total_all   = total_train + total_test
    out.loc["GESAMT"] = [
        total_train, total_test, total_all,
        round(total_train / total_all * 100, 1),
        round(total_test  / total_all * 100, 1),
    ]
    return out

print("Split-Verteilung pro Tag (Mess-IDs):")
print(split_distribution(df_train, df_test, level="ID"))
print()
print("Split-Verteilung pro Tag (Zeilen):")
print(split_distribution(df_train, df_test, level="rows"))

Split-Verteilung pro Tag (Mess-IDs):
           Train_IDs  Test_IDs  Total  Train_%  Test_%
Tag                                                   
Auto           117.0      27.0  144.0     81.2    18.8
Laufen         105.0      31.0  136.0     77.2    22.8
Lift           100.0      18.0  118.0     84.7    15.3
Roundkick       59.0      15.0   74.0     79.7    20.3
Treppe          75.0      32.0  107.0     70.1    29.9
Velo           105.0      14.0  119.0     88.2    11.8
Zug            103.0      26.0  129.0     79.8    20.2
GESAMT         664.0     163.0  827.0     80.3    19.7

Split-Verteilung pro Tag (Zeilen):
           Train_Rows  Test_Rows      Total  Train_%  Test_%
Tag                                                         
Auto         265472.0    50734.0   316206.0     84.0    16.0
Laufen       216707.0    57871.0   274578.0     78.9    21.1
Lift          71119.0    13392.0    84511.0     84.2    15.8
Roundkick     20322.0     8869.0    29191.0     69.6    30.4
Treppe     

Die IDs der jeweiligen Aufnahmen musste mitgegeben werden, damit beim Deep Learning die Cross Validation nach Aufnahmen geteilt werden können. 

In [5]:
def create_sliding_windows_full(df, feature_cols, label_col="Tag",
                                window_size=100, step_size=50):
    X, y, ids = [], [], []

    for id_, group in df.groupby("ID"):
        group = group.sort_values("time_elapsed")
        data  = group[feature_cols].values
        label = group[label_col].iloc[0]

        for start in range(0, len(group) - window_size + 1, step_size):
            X.append(data[start:start + window_size])
            y.append(label)
            ids.append(id_)   #  ID pro Window merken

    return np.array(X), np.array(y), np.array(ids)

In [6]:
feature_cols = [
    "x_acc", "y_acc", "z_acc",
    "x_gyr", "y_gyr", "z_gyr",
    "qx", "qy", "qz", "qw",
    "roll", "pitch", "yaw"
]

WINDOW_SIZE = 100
STEP_SIZE = 50

X_train, y_train, ids_train = create_sliding_windows_full(df_train, feature_cols,
                                               window_size=WINDOW_SIZE,
                                               step_size=STEP_SIZE)

X_test, y_test, ids_test = create_sliding_windows_full(df_test, feature_cols,
                                             window_size=WINDOW_SIZE,
                                             step_size=STEP_SIZE)

In [7]:
np.savez_compressed("../Model_data/train_split1_100WS_cv.npz", X=X_train, y=y_train, ids=ids_train)
np.savez_compressed("../Model_data/test_split1_100WS_cv.npz", X=X_test, y=y_test, ids=ids_test)